# 🎙️ Audio Transcription v1 — faster-whisper (No Speaker Labels)

**Runtime:** Google Colab — T4 GPU  
**What it does:** Transcribes audio to text with timestamps  
**What it does not do:** Speaker identification (see v2 for that)

---
### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your audio file to Google Drive
3. Update `AUDIO_FILE` path in Cell 3

## 1. Install

In [ ]:
!pip install -q faster-whisper

## 2. Mount Google Drive
Required for reliable large file uploads (direct browser upload can truncate files >~50MB)

In [ ]:
from google.colab import drive
drive.mount('/drive')
print("✅ Drive mounted")

## 3. Load model & transcribe

In [ ]:
import time, warnings
from pathlib import Path
from faster_whisper import WhisperModel

warnings.filterwarnings("ignore")

# ── Config ───────────────────────────────────────────────
AUDIO_FILE  = "/drive/MyDrive/your_audio_file.mp3"  # ← update
MODEL_SIZE  = "base"    # tiny | base | small | medium | large-v3
LANGUAGE    = None      # e.g. "en" — None to auto-detect
TIMESTAMPS  = True
# ─────────────────────────────────────────────────────────

audio_path = Path(AUDIO_FILE)
assert audio_path.exists(), f"File not found: {audio_path}"
print(f"File size: {audio_path.stat().st_size / 1024**2:.1f} MB")

print(f"Loading '{MODEL_SIZE}' model...")
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

print("Transcribing...")
t0 = time.time()
options = {"language": LANGUAGE} if LANGUAGE else {}
segments, info = model.transcribe(str(audio_path), **options)
segments = list(segments)

print(f"✅ Done in {time.time()-t0:.1f}s — language: {info.language}")

## 4. View transcript

In [ ]:
def fmt(s):
    return f"{int(s//60):02d}:{int(s%60):02d}"

for seg in segments:
    if TIMESTAMPS:
        print(f"[{fmt(seg.start)} → {fmt(seg.end)}]  {seg.text.strip()}")
    else:
        print(seg.text.strip())

## 5. Save & download

In [ ]:
from google.colab import files

out_path = "/content/transcript.txt"

with open(out_path, "w", encoding="utf-8") as f:
    for seg in segments:
        if TIMESTAMPS:
            f.write(f"[{fmt(seg.start)} → {fmt(seg.end)}]  {seg.text.strip()}\n")
        else:
            f.write(seg.text.strip() + "\n")

print(f"✅ Saved")
files.download(out_path)